### Final Project CSE 476
#### Group members: Neil Shenoy, Kavish Sharma, Jenna Skabelund

In [4]:
%pip install requests python-dotenv

import os, json, textwrap, re, time
import requests

API_KEY = "sk-THSHSCeQUCzILV98j-3xCQ"
API_BASE = os.getenv("API_BASE", "https://openai.rc.asu.edu/v1")  
MODEL    = os.getenv("MODEL_NAME", "qwen3-30b-a3b-instruct-2507")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer—no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 128,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Chain of thought prompting
def CotPrompting(inp: str):
    CotResponse = call_model_chat_completions(
        inp, 
        "You are a helpful assistant. Answer the question while thinking step by step, and explaining your reasoning."
    )
    return CotResponse["text"].strip()

# test
x = CotPrompting("What is 17 * 26? Do not use latex symbols")
print(x)

AttributeError: 'NoneType' object has no attribute 'strip'

In [ ]:
# Self consistency
def SelfConsistency(inp):
    resp1 = call_model_chat_completions(inp)["text"].strip()
    resp2 = call_model_chat_completions(inp)["text"].strip()
    resp3 = call_model_chat_completions(inp)["text"].strip()
    resp4 = call_model_chat_completions(inp)["text"].strip()
    resp5 = call_model_chat_completions(inp)["text"].strip()

    finalResp = call_model_chat_completions(
        f"Here are the 5 responses I got when calling you earlier for question {inp}, which is the most consistent? \
            {resp1}, {resp2}, {resp3}, {resp4}, {resp5}",
        "You are a helpful LLM, give the most consistent answer. Do not explain too much"
    )

    return finalResp["text"].strip()

# test
y = SelfConsistency("What is 123 * 412?")
print(y)

In [ ]:
# Best Of N
def BestOfN(inp: str, n: int):
    resp = []
    for i in range(n):
        resp.append(call_model_chat_completions(inp)["text"].strip()) 
    
    finalResp = call_model_chat_completions(
        f"Here are the {n} responses I got when calling you earlier for question {inp}. Which is the best answer? {resp}",
        "You are a helpful LLM that will give me the best answer. Don't explain too much."
    )
    
    return finalResp["text"].strip()

# test
print(BestOfN("What is the capital of Syria?", 3))

In [ ]:
#Helper
def get_response_text(response_dict):
    if not isinstance(response_dict, dict):
        return ""
    return (response_dict.get("text") or "").strip()

In [ ]:
#ask for an answer, then ask the model "is this answer correct? if not, fix it." Two API calls total.

def solve_with_reflection(question, model=MODEL, temperature=0.0, too_wordy=False):

    first_system = (
        "You are great at finding the correct answers to any given questions. "
        "Solve the user's question. Return only the final answer. Do not explain."
    )

    first_prompt = f"""Question: {question} What is the answer?"""
    first_response = call_model_chat_completions(
        first_prompt,
        system=first_system,
        model=model,
        temperature=temperature,
    )
    initial_answer = get_response_text(first_response)


    reflection_system = (
        "You are great at reviewing questions. "
        "Check whether the proposed answer is correct and complete. "
        "If it is wrong, fix it. "
        "Return only the correct and complete final answer."
    )

    reflection_prompt = f"""You are reviewing a previous answer. Original question: {question} Previous answer: {initial_answer} Task: Check the answer carefully. If it is correct, keep it. If it is wrong or incomplete, fix it. Return the corrected final answer only."""

    reflection_response = call_model_chat_completions(
        reflection_prompt,
        system=reflection_system,
        model=model,
        temperature=0.0,
    )
    
    reflected_answer = get_response_text(reflection_response)
    final_answer = reflected_answer if reflected_answer else initial_answer

    if too_wordy:
        print("QUESTION:", question)
        print("INITIAL ANSWER:", initial_answer)
        print("NEW REFLECTED ANSWER:", reflected_answer)
        print("FINAL ANSWER:", final_answer)

    return {
        "initial_answer": initial_answer,
        "reflected_answer": reflected_answer,
        "final_answer": final_answer,
    }

In [ ]:
#Test

example_question = "What is 17 * 26?"
reflection_result = solve_with_reflection(example_question, too_wordy=True)

In [ ]:
#ask the model to break the question into smaller sub-questions, answer each one, then combine them into a final answer

def solve_with_step_decomposition(question, model=MODEL, temperature=0.0, too_wordy=False):

    decomposition_system = (
        "You are good at breaking problems into smaller steps. "
        "Break the user's question into clear sub-questions or steps. "
        "Do not solve the problem yet."
    )

    decomposition_prompt = f"""Question:{question} Break this question into smaller steps needed to solve it."""

    decomposition_response = call_model_chat_completions(
        decomposition_prompt,
        system=decomposition_system,
        model=model,
        temperature=temperature,
    )

    steps = get_response_text(decomposition_response)


    solve_system = (
        "You are a careful problem solver who always gets the correct question by breaking it down. "
        "Use the provided steps to solve the question. "
        "Return only the final answer."
    )

    solve_prompt = f"""Original question:{question} Steps to follow: {steps} Now solve the original question. Return only the final answer."""

    solve_response = call_model_chat_completions(
        solve_prompt,
        system=solve_system,
        model=model,
        temperature=temperature,
    )

    final_answer = get_response_text(solve_response)


    if too_wordy:
        print("QUESTION:", question)
        print("DECOMPOSED STEPS:", steps)
        print("FINAL ANSWER:", final_answer)

    return {
        "steps": steps,
        "final_answer": final_answer,
    }

In [ ]:
#Test

example_question = "How many minutes are in a year?"
step_result = solve_with_step_decomposition(example_question, too_wordy=True)

In [ ]:
#Helper


def grader_check(question, proposed_answer, model=MODEL):
    grader_system = (
        "You are a grader that is very strict. "
        "Check whether the proposed answer correctly answers the question. "
        "Reply with only YES or NO."
    )

    grader_prompt = f"""Question:{question} Proposed answer: {proposed_answer} Is the proposed answer correct? Reply only YES or NO."""

    grader_response = call_model_chat_completions(
        grader_prompt,
        system=grader_system,
        model=model,
        temperature=0.0,
    )

    grader_text = get_response_text(grader_response).upper()

    return grader_text.startswith("YES")

In [ ]:
# if the model's answer fails your grader check, send it back with "that was wrong, try again" and loop until it gets it right or hits a max attempt limit.


def solve_with_iterative_refinement(question, model=MODEL, temperature=0.0, max_attempts=3, too_wordy=False):

    current_answer = ""
    attempt_history = []

    for attempt in range(1, max_attempts + 1):

        if attempt == 1:
            prompt = f"""Question:{question} Solve the question. Return only the final answer."""

        else:
            prompt = f"""Question:{question} Your previous answer was:{current_answer} That answer was marked wrong. Try again carefully. Return only the final answer."""

        answer_response = call_model_chat_completions(
            prompt,
            system="You are great at finding the correct answer. Return only the final answer.",
            model=model,
            temperature=temperature,
        )

        current_answer = get_response_text(answer_response)

        is_correct = grader_check(question, current_answer, model=model)

        attempt_history.append({
            "attempt": attempt,
            "answer": current_answer,
            "correct": is_correct,
        })

        if too_wordy:
            print("ATTEMPT:", attempt)
            print("ANSWER:", current_answer)
            print("CORRECT:", is_correct)
            print()

        if is_correct:
            break

    return {
        "final_answer": current_answer,
        "attempts": attempt_history,
    }

In [ ]:
#Test

example_question = "What is 17 * (26 + 3 * 5)?"
refinement_result = solve_with_iterative_refinement(example_question, max_attempts=3, too_wordy=True)

In [ ]:
# LLM-as-a-Judge Implementation

def llm_judge(question: str, prediction: str, expected_answer: str, model=MODEL) -> bool:
    
    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."

    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False"""

    response = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (response.get("text") or "").strip().lower()

    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: normalized string match if model gives unexpected output
    normalize = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return normalize(prediction) == normalize(expected_answer)

In [ ]:
#LLM-as-a-Judge Test Cases

test_cases = [
    #True 
    {
        "question": "What is the capital of France?",
        "prediction": "Paris",
        "expected": "Paris",
        "label": "exact match → True"
    },
    #True 
    {
        "question": "What is 3 + 4?",
        "prediction": "The answer is 7.",
        "expected": "7",
        "label": "wordy but correct → True"
    },
    #False
    {
        "question": "What is the capital of France?",
        "prediction": "Berlin",
        "expected": "Paris",
        "label": "wrong answer → False"
    },
    #True
    {
        "question": "What is 17 * 26?",
        "prediction": "442",
        "expected": "442",
        "label": "math correct → True"
    },
    #False
    {
        "question": "What is 17 * 26?",
        "prediction": "452",
        "expected": "442",
        "label": "math wrong → False"
    },
    #True
    {
        "question": "Does ice float on water?",
        "prediction": "Yes, ice floats because it is less dense than liquid water.",
        "expected": "Yes",
        "label": "correct but verbose → True"
    },
]

print("LLM-as-a-Judge Test Results")

all_passed = True
for tc in test_cases:
    result = llm_judge(tc["question"], tc["prediction"], tc["expected"])
    expected_bool = "True" in tc["label"]
    passed = result == expected_bool
    all_passed = all_passed and passed
    status = "PASS" if passed else "FAIL"
    print(f"{status}  [{tc['label']}]")
    print(f"       got: {result}")

print("All tests passed" if all_passed else "Test(s) failed")

In [ ]:
# Tool Use Implementation (2-Step)

def solve_with_tool_use(question: str, model=MODEL) -> dict:

    #Step 1
    tool_decision_system = (
        "You are a reasoning assistant. Your job is to decide if a question "
        "requires arithmetic calculation. "
        "If yes, extract a single Python-evaluable math expression (ie. 17*26, (3+5)/2). "
        "If no, reply with NONE. "
        "Reply in exactly this format:\n"
        "USE_TOOL: <expression or NONE>"
    )

    tool_decision_response = call_model_chat_completions(
        question,
        system=tool_decision_system,
        model=model,
        temperature=0.0,
    )

    decision_text = (tool_decision_response.get("text") or "").strip()

    expression = None
    match = re.search(r"USE_TOOL:\s*(.+)", decision_text, re.IGNORECASE)
    if match:
        candidate = match.group(1).strip()
        if candidate.upper() != "NONE":
            expression = candidate

    #Step 2a - eval
    if expression:
        try:
            allowed = {"__builtins__": {}}
            import math
            allowed.update(vars(math))
            tool_result = eval(expression, allowed)
            return {
                "used_tool": True,
                "expression": expression,
                "final_answer": str(tool_result),
            }
        except Exception as e:
            pass

    #Step 2b - direct
    direct_response = call_model_chat_completions(
        question,
        system="You are a helpful assistant. Reply with only the final answer—no explanation.",
        model=model,
        temperature=0.0,
    )

    return {
        "used_tool": False,
        "expression": expression,
        "final_answer": (direct_response.get("text") or "").strip(),
    }

In [ ]:
#Tool Use Tests

tool_tests = [
    #eval
    {
        "question": "What is 17 * 26?",
        "expected": "442",
        "label": "basic multiplication → use tool",
        "expect_tool": True,
    },
    #eval
    {
        "question": "What is 17 * (26 + 3 * 5)?",
        "expected": "697",
        "label": "nested arithmetic → use tool",
        "expect_tool": True,
    },
    #eval
    {
        "question": "What is 144 / 12?",
        "expected": "12",
        "label": "division → use tool",
        "expect_tool": True,
    },
    #direct
    {
        "question": "What is the capital of Japan?",
        "expected": "Tokyo",
        "label": "factual question → no tool",
        "expect_tool": False,
    },
    #direct
    {
        "question": "Which is heavier, a pound of feathers or a pound of gold?",
        "expected": "They weigh the same",
        "label": "common sense → no tool",
        "expect_tool": False,
    },
]

print("Tool Use Test Results")

for tc in tool_tests:
    result = solve_with_tool_use(tc["question"])
    answer_correct = llm_judge(tc["question"], result["final_answer"], tc["expected"])
    tool_correct = result["used_tool"] == tc["expect_tool"]

    answer_status = "Pass" if answer_correct else "Fail"
    tool_status   = "Pass" if tool_correct   else "Fail"

    print(f"{answer_status} Answer  {tool_status} Tool  [{tc['label']}]")
    print(f"        used_tool:  {result['used_tool']}")
    if result["expression"]:
        print(f"        expression: {result['expression']}")
    print(f"        answer:     {result['final_answer']}")
    print()

In [6]:
#Step 3: solve()
def solve(question: str, model=MODEL) -> str:

    #domain classification
    classify_system = (
        "You are a question classifier. "
        "Reply with exactly one word from this list: "
        "math, coding, planning, common_sense, future_prediction. "
        "No punctuation. No explanation."
    )
    classify_response = call_model_chat_completions(
        question,
        system=classify_system,
        model=model,
        temperature=0.0,
    )
    domain = (classify_response.get("text") or "").strip().lower()

    #sanitize
    valid_domains = {"math", "coding", "planning", "common_sense", "future_prediction"}
    if domain not in valid_domains:
        domain = "fallback"

    candidates = []

    #techniques selected based on domain

    if domain == "math":
        tool_result = solve_with_tool_use(question, model=model)
        candidates.append(tool_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        refinement_result = solve_with_iterative_refinement(
            question, model=model, max_attempts=3
        )
        candidates.append(refinement_result["final_answer"])

    elif domain == "coding":
        decomp_result = solve_with_step_decomposition(question, model=model)
        candidates.append(decomp_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        refinement_result = solve_with_iterative_refinement(
            question, model=model, max_attempts=3
        )
        candidates.append(refinement_result["final_answer"])

    elif domain == "planning":
        decomp_result = solve_with_step_decomposition(question, model=model)
        candidates.append(decomp_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    elif domain == "common_sense":
        bon_answer = BestOfN(question, n=3)
        candidates.append(bon_answer)

        sc_answer = SelfConsistency(question)
        candidates.append(sc_answer)

    elif domain == "future_prediction":
        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    else:
        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    #Delete duplicate candidates
    seen = set()
    unique_candidates = []
    for c in candidates:
        normalized = re.sub(r"\s+", " ", (c or "").strip().lower())
        if normalized and normalized not in seen:
            seen.add(normalized)
            unique_candidates.append(c)

    if len(unique_candidates) == 1:
        return unique_candidates[0]

    #LLM-as-a-Judge picks best candidate 
    judge_system = (
        "You are an expert judge selecting the best answer to a question. "
        "Return only the best answer verbatim. No explanation."
    )
    candidates_block = "\n".join(
        f"Candidate {i+1}: {c}" for i, c in enumerate(unique_candidates)
    )
    judge_prompt = (
        f"Question: {question}\n\n"
        f"{candidates_block}\n\n"
        "Which candidate best answers the question? Return only that answer verbatim."
    )
    judge_response = call_model_chat_completions(
        judge_prompt,
        system=judge_system,
        model=model,
        temperature=0.0,
    )
    best = (judge_response.get("text") or "").strip()

    #backup
    return best if best else unique_candidates[0]